In [1]:
!pip install torch torchvision matplotlib

In [2]:
import torch
import torchvision
import torchvision.transforms as transforms

# Transform: resize + convert to tensor
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

# Load CIFAR-10 train dataset
trainset = torchvision.datasets.CIFAR10(
    root='./data',
    train=True,
    download=True,
    transform=transform
)

trainloader = torch.utils.data.DataLoader(
    trainset,
    batch_size=32,
    shuffle=True
)

# Load test dataset
testset = torchvision.datasets.CIFAR10(
    root='./data',
    train=False,
    download=True,
    transform=transform
)

testloader = torch.utils.data.DataLoader(
    testset,
    batch_size=32,
    shuffle=False
)

print("CIFAR-10 loaded successfully!")

100%|██████████| 170M/170M [00:03<00:00, 42.7MB/s]


CIFAR-10 loaded successfully!


In [3]:
import torch.nn as nn
import torch.nn.functional as F

class BaselineCNN(nn.Module):
    def __init__(self):
        super(BaselineCNN, self).__init__()

        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)

        self.pool = nn.MaxPool2d(2, 2)

        self.fc1 = nn.Linear(128 * 28 * 28, 256)
        self.fc2 = nn.Linear(256, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))   # 224 → 112
        x = self.pool(F.relu(self.conv2(x)))   # 112 → 56
        x = self.pool(F.relu(self.conv3(x)))   # 56 → 28

        x = x.view(-1, 128 * 28 * 28)

        x = F.relu(self.fc1(x))
        x = self.fc2(x)

        return x

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = BaselineCNN().to(device)

print("Baseline model created!")

Baseline model created!


In [5]:
class DepthwiseSeparableConv(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super(DepthwiseSeparableConv, self).__init__()

        # Depthwise convolution
        self.depthwise = nn.Conv2d(
            in_channels,
            in_channels,
            kernel_size=3,
            stride=stride,
            padding=1,
            groups=in_channels   # KEY LINE
        )

        # Pointwise convolution
        self.pointwise = nn.Conv2d(
            in_channels,
            out_channels,
            kernel_size=1
        )

        self.bn = nn.BatchNorm2d(out_channels)

    def forward(self, x):
        x = self.depthwise(x)
        x = self.pointwise(x)
        x = self.bn(x)
        return F.relu(x)

In [6]:
class MobileNet(nn.Module):
    def __init__(self, num_classes=10):
        super(MobileNet, self).__init__()

        # First standard conv layer
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, stride=2, padding=1)
        self.bn1 = nn.BatchNorm2d(32)

        # Depthwise separable layers
        self.layers = nn.Sequential(
            DepthwiseSeparableConv(32, 64, stride=1),
            DepthwiseSeparableConv(64, 128, stride=2),
            DepthwiseSeparableConv(128, 128, stride=1),
            DepthwiseSeparableConv(128, 256, stride=2),
            DepthwiseSeparableConv(256, 256, stride=1),
            DepthwiseSeparableConv(256, 512, stride=2),

            # 5 repeated layers
            DepthwiseSeparableConv(512, 512, stride=1),
            DepthwiseSeparableConv(512, 512, stride=1),
            DepthwiseSeparableConv(512, 512, stride=1),
            DepthwiseSeparableConv(512, 512, stride=1),
            DepthwiseSeparableConv(512, 512, stride=1),

            DepthwiseSeparableConv(512, 1024, stride=2),
            DepthwiseSeparableConv(1024, 1024, stride=1)
        )

        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(1024, num_classes)

    def forward(self, x):
        x = F.relu(self.bn1(self.conv1(x)))
        x = self.layers(x)

        x = self.avgpool(x)
        x = x.view(x.size(0), -1)

        x = self.fc(x)
        return x

In [7]:
mobilenet = MobileNet().to(device)

print("MobileNet model created!")

MobileNet model created!


In [8]:
import torch.optim as optim

# Loss function
criterion = nn.CrossEntropyLoss()

# Optimizers
optimizer_baseline = optim.Adam(model.parameters(), lr=0.001)
optimizer_mobilenet = optim.Adam(mobilenet.parameters(), lr=0.001)

print("Loss and optimizers ready!")

Loss and optimizers ready!


In [9]:
def train_model(model, optimizer, trainloader, epochs=2):
    model.train()

    for epoch in range(epochs):
        running_loss = 0.0

        for images, labels in trainloader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()

            outputs = model(images)
            loss = criterion(outputs, labels)

            loss.backward()
            optimizer.step()

            running_loss += loss.item()

        print(f"Epoch {epoch+1}, Loss: {running_loss/len(trainloader):.4f}")

In [10]:
print("Training Baseline CNN...")
train_model(model, optimizer_baseline, trainloader, epochs=2)

Training Baseline CNN...
Epoch 1, Loss: 1.5297
Epoch 2, Loss: 1.2325


In [11]:
print("Training MobileNet...")
train_model(mobilenet, optimizer_mobilenet, trainloader, epochs=2)

Training MobileNet...
Epoch 1, Loss: 1.2611
Epoch 2, Loss: 0.7534


In [12]:
def evaluate(model, testloader):
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in testloader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / total
    return accuracy

In [13]:
baseline_acc = evaluate(model, testloader)
mobilenet_acc = evaluate(mobilenet, testloader)

print(f"Baseline CNN Accuracy: {baseline_acc:.2f}%")
print(f"MobileNet Accuracy: {mobilenet_acc:.2f}%")

Baseline CNN Accuracy: 56.46%
MobileNet Accuracy: 77.24%


In [14]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters())

baseline_params = count_parameters(model)
mobilenet_params = count_parameters(mobilenet)

print(f"Baseline CNN Parameters: {baseline_params}")
print(f"MobileNet Parameters: {mobilenet_params}")

Baseline CNN Parameters: 25786186
MobileNet Parameters: 3218250


We implemented a baseline CNN and MobileNet from scratch and evaluated both models on CIFAR-10.

The baseline CNN achieved an accuracy of 56.46% with approximately 25.7 million parameters. In contrast, MobileNet achieved a significantly higher accuracy of 77.24% while using only 3.2 million parameters.

This demonstrates that MobileNet is not only computationally efficient but also more effective in learning representations for this dataset. The use of depthwise separable convolutions reduces model complexity while maintaining strong performance.

Thus, our results align with the original paper’s claim that MobileNet provides an excellent trade-off between accuracy and efficiency.

In [15]:
results_text = f"""
Baseline CNN Accuracy: {baseline_acc:.2f}%
MobileNet Accuracy: {mobilenet_acc:.2f}%

Baseline CNN Parameters: {baseline_params}
MobileNet Parameters: {mobilenet_params}
"""

with open("results.txt", "w") as f:
    f.write(results_text)

print("Results saved to results.txt")

Results saved to results.txt


In [16]:
from google.colab import files
files.download("results.txt")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>